## Anvio metagenomes cheatsheet, **after having done QC and assembly**

**IMPORTANT:** This is NOT the first step in your metagenomics analysis. You should already have 1) QC'd fastqs and 2) an assembly via spades or megahit or something. Also, this is moderately advanced technically and details matter. If things (`screen`, port forwarding, conda environments, for loops, `()&` notation, contig lengths) don't make sense, **STOP and ask/look up until comfortable**. **This is not a tutorial** or a turn the crank script. It is a framework of common steps in a typical order. Do not expect scientific outcomes if you just run each step in order. A-OK to not understand your first time, but use this as an opportunity to learn. **THIS IS NOT A PROTOCOL.**

This notebook runs in any environment with jupyter, e.g. `conda activate read_mapping` Make sure you are port forwarding the port Jupyter will use. Running this notebook inside a screen is convenient.

This notebook does not actually *run* any analyses. It just generates the code for you to paste into a seprate terminal (ideally in a `screen`). Or for better documentation, paste into some `00_formatcontigs.sh` then run that. **All commands need to be run separately in an** `anvio` **environment, e.g.** `conda activate anvio-9`

This script should be top-level in your working metagenomes folder. for example, `/data1/projects/xenos/metagenomes/anvio_example.ipynb`.
Parallel with this script should be an assembly (contigs) in a folder called like `assemblies/spades_SOMETHING` For example, at the end of this, your folder should look like this: 
```
metagenomes
├── anvio_example.ipynb
├── assemblies
│   ├── megahit_coXenos
│   └── spades_xeno_coassembly
├── contigs
│   ├── xeno_comega-2500-CONTIGS.db
│   ├── xeno_comega-2500-formatted.fna
│   └── xeno_comega-2500-report.txt
├── fastq_manifest.csv
├── mapping
│   ├── VJO_11317_xeno_costarica
│   ├── VJO_11317_xeno_costarica.bam
│   ├── VJO_11317_xeno_costarica.bam.bai
│   ├── VJO_1925_xeno_alaska
│   ├── VJO_1925_xeno_alaska.bam
│   ├── VJO_1925_xeno_alaska.bam.bai
│   ├── xeno_comega-2500.1.bt2
│   ├── xeno_comega-2500.2.bt2
│   ├── ...
└── xeno_comega-2500-MERGED
    ├── AUXILIARY-DATA.db
    ├── PROFILE.db
    └── RUNLOG.txt
 ```

In [8]:
import re, glob

In [9]:
## CHANGE ALL OF THESE
prefix="megahit19800"  # cannot start with a number, ideally one word
contiglen="2500"

#spadesdir="assemblies/spades_VJO_15581B-QUALITY_PASSED/contigs.fasta"         # SPADES STYLE
spadesdir="assemblies/megahit_19800/final.contigs.fa"    # MEGAHIT STYLE

readpath="/export/data1/data/metagenomes/19800_worms/iu_filter_minoche/"

fqs=sorted(glob.glob(readpath+"/*fastq*")) # probably don't need to change this unless you use .fq or something

In [10]:
## generic paraemters
cpus=8

#### If you don't have a `fastq_manifest.csv` made, this will make it for you
Also OK to edit manually. One sample (fastq pair) per line, format of 
```
/data1/data/metagenomes/CORE/iu_filter_minoche/SAMPLE1_R1.fastq,/data1/data/metagenomes/CORE/iu_filter_minoche/SAMPLE1_R2.fastq
/data1/data/metagenomes/CORE/iu_filter_minoche/SAMPLE2_R1.fastq,/data1/data/metagenomes/CORE/iu_filter_minoche/SAMPLE2_R2.fastq
```
and so on. Obviously, don't run this next block if you manually make this file.

In [5]:
with open('fastq_manifest.csv','w') as f:
    for x in list(zip(fqs[::2],fqs[1::2])):
        f.write(x[0] + ',' + x[1])
        f.write('\n')


# Generating code to paste
Fast but always dangerous. check it matches, and is updating variables right

## World's shortest intro to anvi'o
This doc uses [anvio](https://anvio.org/) heavily. You can absolutely run this code, have no errors, and end up with completely  flawed output. **Please** don't just run this without having an idea of a) expectations and b) appropriate sanity-checks. Please consider spending some time reading over best-practice examples and some examples of unintentional pitfalls:
* [The infamous very detailed overview](https://merenlab.org/tutorials/infant-gut/)
* [A more bite-sized overview to get you thinking about binning realities](https://anvio.org/tutorials/binning-popgen-oral-microbiome/) 
* [Read recruitment](https://merenlab.org/tutorials/read-recruitment/)
* [Outdated specifics (pre-MIMAG) but the underlying problem is constant](https://merenlab.org/2016/06/09/assessing-completion-and-contamination-of-MAGs/) and [real-world example HERE](https://merenlab.org/2015/06/25/screening-cultivars/)

### Stage 1: Cleanly formating and initializing db

anvi-script-reformat does the renaming **AND LENGTH FILTERING** see parameter in the opening of this notebook. Different tools have different tolerances, so best practice is enforce `[A-Z0-9_- ]` from the start

anvi-gen-contigs stores the contig info in a useable way to merge with all the gene/contig level info later. The critical step here is prodigal is run to call genes (in metagenome mode). Adding more cpus scales this step, adjust above before generating pastable code
  
`anvi-run-hmms` identifies single copy protein genes for bacteria, archaea (think GTDB) and \[tr\]RNAs. Running the HMMs is ESSENTIAL for estimating bin completion later. Technically optional, but not really.

In [5]:
print("mkdir -p contigs\n\
anvi-script-reformat-fasta -l {CONTIGLEN} -r contigs/{PREFIX}-{CONTIGLEN}-report.txt --simplify-names -o contigs/{PREFIX}-{CONTIGLEN}-formatted.fna --prefix {PREFIX} {SPADESDIR}\n\
anvi-gen-contigs-database -f contigs/{PREFIX}-{CONTIGLEN}-formatted.fna -T {CPUS} -o contigs/{PREFIX}-{CONTIGLEN}-CONTIGS.db\n\
anvi-run-hmms -T {CPUS} -c contigs/{PREFIX}-{CONTIGLEN}-CONTIGS.db\n\
".format(CONTIGLEN=contiglen, PREFIX=prefix, SPADESDIR=spadesdir, CPUS=cpus))

mkdir -p contigs
anvi-script-reformat-fasta -l 2500 -r contigs/megahit19800-2500-report.txt --simplify-names -o contigs/megahit19800-2500-formatted.fna --prefix megahit19800 assemblies/megahit_19800/final.contigs.fa
anvi-gen-contigs-database -f contigs/megahit19800-2500-formatted.fna -T 8 -o contigs/megahit19800-2500-CONTIGS.db
anvi-run-hmms -T 8 -c contigs/megahit19800-2500-CONTIGS.db



### Stage 2: Mapping
This can happen in parallel wth the `anvi-gen-contigs-database` **after** the `anvi-script-reformat` is done (this can be quick relative to how long contigs db / prodigal can take)

key steps: build a mappable database **identical** to what went into anvio contigs db (this is why anvi-reformat first). We
map each of your fastq files separately onto the entirety of what went into contigs db. Keep only reads that map, keep as `.bam` to be efficient spacewise. 

In [6]:
print("mkdir -p mapping\n\
bowtie2-build contigs/{PREFIX}-{CONTIGLEN}-formatted.fna mapping/{PREFIX}-{CONTIGLEN} --threads 1\n\
\n\
for r1r2 in $(cat fastq_manifest.csv); do\n\
 (r1=$(echo $r1r2 | cut -d, -f 1); r2=$(echo $r1r2 | cut -d, -f 2);\n\
 pref=$(basename $r1 | sed 's/-QUALITY.*$//');\n\
 (bowtie2 -x mapping/{PREFIX}-{CONTIGLEN} -1 $r1 -2 $r2 -p 2 --no-unal) 2>mapping/$pref.txt | samtools view -b -@ 2 | samtools sort -@ 2 - > mapping/$pref.bam \n\
 samtools index mapping/$pref.bam)& \n\
done\n\
".format(CONTIGLEN=contiglen, PREFIX=prefix, SPADESDIR=spadesdir, CPUS=cpus))

mkdir -p mapping
bowtie2-build contigs/megahit19800-2500-formatted.fna mapping/megahit19800-2500 --threads 1

for r1r2 in $(cat fastq_manifest.csv); do
 (r1=$(echo $r1r2 | cut -d, -f 1); r2=$(echo $r1r2 | cut -d, -f 2);
 pref=$(basename $r1 | sed 's/-QUALITY.*$//');
 (bowtie2 -x mapping/megahit19800-2500 -1 $r1 -2 $r2 -p 2 --no-unal) 2>mapping/$pref.txt | samtools view -b -@ 2 | samtools sort -@ 2 - > mapping/$pref.bam 
 samtools index mapping/$pref.bam)& 
done




### MUST HAVE CONTIGS DB FINISHED BEFORE THIS STEP

Also note that `-M` flag in `anvi-profile` sets minimum contig length to profile. Default is 1000, can specify `0` just make sure it is not larger than what you set as `contiglen` before or you will get weird resuls.

In [11]:
print("mkdir -p mapping\n\
for r1r2 in $(cat fastq_manifest.csv); do\n\
 r1=$(echo $r1r2 | cut -d, -f 1); r2=$(echo $r1r2 | cut -d, -f 2);\n\
 pref=$(basename $r1 | sed 's/-QUALITY.*$//');\n\
 anvi-profile -c contigs/{PREFIX}-{CONTIGLEN}-CONTIGS.db -M 1000 -T {CPUS} -o mapping/$pref -i mapping/$pref.bam\n\
done\n\
\n\
anvi-merge -c contigs/{PREFIX}-{CONTIGLEN}-CONTIGS.db -o {PREFIX}-{CONTIGLEN}-MERGED mapping/*/PROFILE.db\n\
\n\
".format(CONTIGLEN=contiglen, PREFIX=prefix, SPADESDIR=spadesdir, CPUS=cpus))

mkdir -p mapping
for r1r2 in $(cat fastq_manifest.csv); do
 r1=$(echo $r1r2 | cut -d, -f 1); r2=$(echo $r1r2 | cut -d, -f 2);
 pref=$(basename $r1 | sed 's/-QUALITY.*$//');
 anvi-profile -c contigs/megahit19800-2500-CONTIGS.db -M 1000 -T 8 -o mapping/$pref -i mapping/$pref.bam
done

anvi-merge -c contigs/megahit19800-2500-CONTIGS.db -o megahit19800-2500-MERGED mapping/*/PROFILE.db




### Binning. The good stuff. 
Typically want to run a few methods. Best results with multiple metagenomes (e.g. fastq pairs) mapped. 

first one `CONCOCT` needs a number of clusters to aim for, initial guess use expected number of bins divided by 1.5 or 2

second one `METABAT` needs minimum size `-m`. 2500 is jgi standard (i think), 1500 is anvio more generous default

Last one `DASTOOL` needs to be last - this is a meta-binner that attempts to make consensus better bins by pooling other binners. Need to specificy which existing collections of bins to use.

In [13]:
print("\
anvi-cluster-contigs -c contigs/{PREFIX}-{CONTIGLEN}-CONTIGS.db -p {PREFIX}-{CONTIGLEN}-MERGED/PROFILE.db -T {CPUS} --driver concoct -C CONCOCT --clusters 50 --just-do-it \n\
anvi-cluster-contigs -c contigs/{PREFIX}-{CONTIGLEN}-CONTIGS.db -p {PREFIX}-{CONTIGLEN}-MERGED/PROFILE.db -T {CPUS} --driver metabat2 -C METABAT2 -m 1500 --just-do-it \n\
anvi-cluster-contigs -c contigs/{PREFIX}-{CONTIGLEN}-CONTIGS.db -p {PREFIX}-{CONTIGLEN}-MERGED/PROFILE.db -T {CPUS} --driver maxbin2 -C MAXBIN2 --just-do-it \n\
\n\
anvi-cluster-contigs -c contigs/{PREFIX}-{CONTIGLEN}-CONTIGS.db -p {PREFIX}-{CONTIGLEN}-MERGED/PROFILE.db -T {CPUS} --driver dastool --search-engine diamond -C DASTOOL -S CONCOCT,METABAT2,MAXBIN2 --just-do-it \n\
".format(CONTIGLEN=contiglen, PREFIX=prefix, SPADESDIR=spadesdir, CPUS=cpus))

anvi-cluster-contigs -c contigs/megahit19800-2500-CONTIGS.db -p megahit19800-2500-MERGED/PROFILE.db -T 8 --driver concoct -C CONCOCT --clusters 50 --just-do-it 
anvi-cluster-contigs -c contigs/megahit19800-2500-CONTIGS.db -p megahit19800-2500-MERGED/PROFILE.db -T 8 --driver metabat2 -C METABAT2 -m 1500 --just-do-it 
anvi-cluster-contigs -c contigs/megahit19800-2500-CONTIGS.db -p megahit19800-2500-MERGED/PROFILE.db -T 8 --driver maxbin2 -C MAXBIN2 --just-do-it 

anvi-cluster-contigs -c contigs/megahit19800-2500-CONTIGS.db -p megahit19800-2500-MERGED/PROFILE.db -T 8 --driver dastool --search-engine diamond -C DASTOOL -S CONCOCT,METABAT2,MAXBIN2 --just-do-it 



## THOU SHALT NOT SKIP MANUAL BIN REFINEMENT
#### Or be haunted by the spirits of Dan & Daan
Seriously, it's worth it. Use `anvi-refine` and follow [EXAMPLE 1](https://merenlab.org/2015/05/11/anvi-refine/) or [EXAMPLE 2](https://merenlab.org/2017/05/11/anvi-refine-by-veronika/)

### Extra stuff:
#### Exporting: `anvi-summarize`
**MUST** include a -C collection id, example here uses DASTOOL but can be any collection. Most useful after making bins WORTH exporting, but can be useful to rank proto-bins to decide which to refine/focus on

#### Rough taxonomy: `anvi-run-scg-taxonomy`

#### Functions: `anvi-run-kegg-kofams` etc


In [14]:
print("\
anvi-run-scg-taxonomy -c contigs/{PREFIX}-{CONTIGLEN}-CONTIGS.db -T {CPUS} \n\
\n\
\n\
#anvi-summarize -c contigs/{PREFIX}-{CONTIGLEN}-CONTIGS.db -p {PREFIX}-{CONTIGLEN}-MERGED/PROFILE.db -C DASTOOL -o {PREFIX}-{CONTIGLEN}-SUMMARY-DASTOOL \n\
\n\
\n\
anvi-run-kegg-kofams -c contigs/{PREFIX}-{CONTIGLEN}-CONTIGS.db -T {CPUS} \n\
anvi-run-pfams -c contigs/{PREFIX}-{CONTIGLEN}-CONTIGS.db  -T {CPUS} \n\
anvi-run-ncbi-cogs -c contigs/{PREFIX}-{CONTIGLEN}-CONTIGS.db  -T {CPUS} \n\
\n\
".format(CONTIGLEN=contiglen, PREFIX=prefix, SPADESDIR=spadesdir, CPUS=cpus))

anvi-run-scg-taxonomy -c contigs/megahit19800-2500-CONTIGS.db -T 8 


#anvi-summarize -c contigs/megahit19800-2500-CONTIGS.db -p megahit19800-2500-MERGED/PROFILE.db -C DASTOOL -o megahit19800-2500-SUMMARY-DASTOOL 


anvi-run-kegg-kofams -c contigs/megahit19800-2500-CONTIGS.db -T 8 
anvi-run-pfams -c contigs/megahit19800-2500-CONTIGS.db  -T 8 
anvi-run-ncbi-cogs -c contigs/megahit19800-2500-CONTIGS.db  -T 8 


